In [1]:
## This layer is typically used to build “Decoder only” models such as ChatGPT, LLama etc.
## we’ll use the Decoder layers to build a Decoder-only model similar to GPT.

In [39]:
import numpy as np
import pandas as pd
import torch
import lightning as L
from copy import deepcopy

In [8]:
# # import datasets
# from torchtext.datasets import IMDB

# train_iter = IMDB(split='train')

# def tokenize(label, line):
#     return line.split()

# tokens = []
# for label, line in train_iter:
#     tokens += tokenize(label, line)
#     break

In [31]:
## Single decoder layers

class DecoderLayer(torch.nn.Module):
    def __init__(self, embed_dim, n_heads, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.mha=torch.nn.MultiheadAttention(embed_dim=embed_dim,num_heads=n_heads,dropout=0.1, batch_first=True)
        self.norm1=torch.nn.LayerNorm(normalized_shape=embed_dim)
        self.norm2=torch.nn.LayerNorm(normalized_shape=embed_dim)
        self.dropout1=torch.nn.Dropout(p=dropout)
        self.dropout2=torch.nn.Dropout(p=dropout)
        self.ff_block=torch.nn.Sequential(torch.nn.Linear(in_features=embed_dim,out_features=dim_feedforward),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=dim_feedforward,out_features=embed_dim)
        )
        
    def forward(self, x: torch.Tensor, key_padding_mask=None, attn_mask=None):
        attn_output, attn_weights = self.mha(
            x, x, x, attn_mask=attn_mask, key_padding_mask=key_padding_mask
        )
        x = self.norm1(x + self.dropout1(attn_output))
        projection = self.ff_block(x)
        x = self.norm2(x + self.dropout2(projection))
        return x

In [32]:
## For loop creating num_layers decoder layers resulting in a Decoder network

class Decoder(torch.nn.Module):
    def __init__(self, decoder_layer, num_layers: int):
        super().__init__()
        layers = []
        for _ in range(num_layers):
            layers.append(deepcopy(decoder_layer))
        self.layers = torch.nn.ModuleList(layers)

    def forward(self, x, key_padding_mask=None, attn_mask=None):
        for layer in self.layers:
            x = layer(x, key_padding_mask=key_padding_mask, attn_mask=attn_mask)
        return x

In [33]:
# Let’s implement a model similar to GPT.
# The model will return probabilities of next token.



## Ordinary encodings plus Positional Encodings

import math
class PositionalEncoding(torch.nn.Module):
    # source: https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html#Positional-encoding
    def __init__(self, embed_dim, max_len=256):
        super().__init__()
        # create a matrix of [seq_len, hidden_dim] representing positional encoding for each token in sequence
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # (max_len, 1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe, persistent=False)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

In [34]:
class TinyGPT(torch.nn.Module):
    def __init__(
        self,
        num_layers: int,
        vocab_size: int,
        embed_dim: int,
        max_len: int,
        n_heads: int,
        dim_feedforward: int,
        pad_token_idx: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=pad_token_idx
        )
        self.positional_encoding = PositionalEncoding(embed_dim=embed_dim, max_len=max_len)
        self.decoders = Decoder(
            decoder_layer=DecoderLayer(
                embed_dim=embed_dim,
                n_heads=n_heads,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
            ),
            num_layers=num_layers,
        )
        self.lm_head = torch.nn.Linear(in_features=embed_dim, out_features=vocab_size)

    def forward(self, input_ids, key_padding_mask=None):
        bs, seq_len = input_ids.size()
        embeddings = self.get_embeddings(input_ids)
        # generate a causal mask
        attn_mask = torch.nn.Transformer.generate_square_subsequent_mask(sz=seq_len, device=input_ids.device)
        embeddings = self.decoders(embeddings, key_padding_mask=key_padding_mask, attn_mask=attn_mask)
        logits = self.lm_head(embeddings)
        return logits

    def get_embeddings(self, input_ids):
        return self.positional_encoding(self.embedding(input_ids))
    
    def get_model_param_count(self):
        return sum(t.numel() for t in self.parameters())
    
    def generate(self, tokenizer, initial_text=None, max_len: int=20):
        device = next(self.parameters()).device
        input_ids = [tokenizer.cls_token_id]
        if initial_text:
            # tokenizer add SEP token at the end, do not include that one
            input_ids = tokenizer(initial_text)['input_ids'][:-1] # type: ignore

        while len(input_ids) < max_len:
            logits = self(input_ids=torch.LongTensor(input_ids).unsqueeze(0).to(device))
            # take the logits of the last token and use a temperature of 0.1
            logits = logits[0][-1] / 0.1
            
            # greedy sampling. take the token with max "probability"
            next_token_id = logits.argmax(dim=-1).item()
            input_ids.append(next_token_id)
            if next_token_id == tokenizer.sep_token_id:
                break

        return tokenizer.decode(input_ids)

In [18]:
## The generate takes the logit produced by last token in the input and then chooses the next token as the token with highest 
# value (also called greedy decoding).

import datasets
from transformers import AutoTokenizer

# can choose other tokenizers as well
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
# let's limit the max number of tokens in a sequence to be 128.
# longer sequences will be truncated
tokenizer.model_max_length = 128
news_ds = datasets.load_dataset("fancyzhx/ag_news", split="train")
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True)
news_ds = news_ds.map(tokenize, batched=True)


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

C:\Users\Amrita\anaconda3\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Amrita\.cache\huggingface\hub\datasets--fancyzhx--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

In [35]:
class DataCollatorForLM:
    def __init__(self, pad_token_idx: int):
        self.pad_token_idx = pad_token_idx
    
    def __call__(self, batch):
        input_ids = []
        # collect the input_ids as torch Tensor 
        for row in batch:
            input_ids.append(torch.LongTensor(row['input_ids']))

        # pad the input_ids so that all of them have same shape
        input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=self.pad_token_idx)
        # any input_ids that is same as pad_token_idx will be considered as key padding mask
        # for a mask, value of True means it will not take part in attention
        key_padding_mask = input_ids == self.pad_token_idx
        # labels will be same as the input_ids
        # we will shift the labels when calculating the loss
        labels = input_ids.clone()
        # we also set the token_id of padded tokens to -100 so that we can ignore these
        # when calculating cross entropy loss because we do not care what the model predicts
        # for these padded tokens
        labels[labels == self.pad_token_idx] = -100
        return {"input_ids": input_ids, "key_padding_mask": key_padding_mask, "labels": labels}

In [52]:
from torch.utils.data import DataLoader
# news_ds = news_ds.train_test_split(test_size=0.2)
bs = 128
collate_fn = DataCollatorForLM(pad_token_idx=tokenizer.pad_token_id)
train_dl = DataLoader(news_ds['train'], batch_size=bs, shuffle=True, collate_fn=collate_fn)
test_dl = DataLoader(news_ds['test'], batch_size=bs, shuffle=False, collate_fn=collate_fn)

In [53]:
class LitTinyGPT(L.LightningModule):
    def __init__(self, gpt: TinyGPT):
        super().__init__()
        self.gpt = gpt

    def compute_loss(self, batch):
        input_ids = batch["input_ids"]
        key_padding_mask = batch["key_padding_mask"]
        labels = batch["labels"]
        logits = self.gpt(input_ids=input_ids, key_padding_mask=key_padding_mask)
        # flatten the labels
        shift_labels = labels[..., 1:].contiguous().view(-1) # 1D array with total elements = bs * (seq_len - 1)

        # shift logits so that we discard the probabilties for the last one
        # since final token does not have next token to predict
        shift_logits = logits[..., :-1, :].contiguous()
        shift_logits = shift_logits.view(-1, shift_logits.size(-1)) # 2D array
        
        # we ignore the predictions for labels which have value of -100 (as specified in the data collator)
        loss = torch.nn.functional.cross_entropy(
            shift_logits, target=shift_labels, ignore_index=-100
        )
        return loss
    
    def training_step(self, batch, batch_idx):        
        loss = self.compute_loss(batch=batch)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss = self.compute_loss(batch=batch)
        self.log_dict({"val_loss": loss, "perplexity": torch.exp(loss)}, on_epoch=True, on_step=True)

    def configure_optimizers(self):
        optim = torch.optim.Adam(params=self.parameters(), lr=1e-3)
        return optim

In [63]:
# 4 decoder layers
num_layers = 2
vocab_size = tokenizer.vocab_size
# embedding size of 512
embed_dim = 16
# 8 heads on MHA
n_heads = 8
dim_feedforward = 125
# max_len is needed by PositionalEmbedding
max_len = tokenizer.model_max_length
gpt = TinyGPT(
    num_layers=num_layers,
    vocab_size=vocab_size,
    max_len=max_len,
    embed_dim=embed_dim,
    n_heads=n_heads,
    dim_feedforward=dim_feedforward,
    pad_token_idx=tokenizer.pad_token_id,
    dropout=0.1,
)
lit_gpt = LitTinyGPT(gpt=gpt)
print(f"Total model parameters = {gpt.get_model_param_count():,}")

Total model parameters = 1,017,812


In [64]:
from lightning import LightningModule, Trainer
class MyCallaback(L.Callback):
    def generate_texts(self, pl_module: LitTinyGPT):
        pl_module.print(pl_module.gpt.generate(tokenizer=tokenizer, initial_text=None, max_len=30))
        pl_module.print()
        pl_module.print(pl_module.gpt.generate(tokenizer=tokenizer, initial_text="france starts", max_len=30))
        pl_module.print()
        pl_module.print(pl_module.gpt.generate(tokenizer=tokenizer, initial_text="vw considers opening", max_len=30))
        pl_module.print("==============")

    def on_train_epoch_end(self, trainer: Trainer, pl_module: LightningModule) -> None:
        self.generate_texts(pl_module=pl_module)

In [65]:
num_epochs = 2
trainer = L.Trainer(fast_dev_run=False, max_epochs=num_epochs, max_steps=-1, log_every_n_steps=20, callbacks=[MyCallaback()])
trainer.fit(lit_gpt, train_dataloaders=train_dl, val_dataloaders=test_dl)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name | Type    | Params | Mode 
-----------------------------------------
0 | gpt  | TinyGPT | 1.0 M  | train
-----------------------------------------
1.0 M     Trainable params
0         Non-trainable params
1.0 M     Total params
4.071     Total estimated model params size (MB)
28        Modules in train mode
0         Modules in eval mode


Sanity Checking: |                                                                               | 0/? [00:00<…

Training: |                                                                                      | 0/? [00:00<…


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined